# Janus-Pro: 为统一多模态模型解耦编码器

统一多模态模型有一个不能避免的张力。对于理解任务来说，需要语义特征，SigLIP 或者 DINOv2 输出富含概念层级信息的向量。对于生产任务来说，需要重建友好的编码，VQ tokens能够生产清晰的像素。这两个目标在一个编码器里面并不适配。Janus 和 Janus-Pro 的解决方案是：将两个编码器解耦，在不同任务间共享一个transformer主体，但是理解的时候路由到SigLIP，生成的时候则路由到VQ 分词器。


## 问题描述

统一模型在理解和生成任务上共享一个transformer主体。以前的尝试（Chanmeleon，Show-o 和 Transfusion）是在两个方向上都适用同一个视觉分词器。这个分词器是一种妥协：
- 对于重建或生成任务优化过： VQ-VAE能够捕捉细粒度的像素细节，但是产生的token之间语义协调度弱。
- 对于语义或理解任务优化过： SigLIP 嵌入将猫的图片推向猫的词元，但是不能保证重建效果好。

Show-o 的 Transfusion 都为其中一个方向付出了视觉质量税。Janus-Pro 的思路是，为什么不同需求的任务不使用多个分词器？

## 基本概念

### 解耦视觉编码

Janus-Pro的架构将两个编码器做了分离。
- 理解。  输入图像--> SigLIP --> 2层MLP --> transformer 主体
- 生成。  输入图像（如果约束在原本的图像之上）--> VQ tokenizer --> token IDs --> transformer 主体
- 输出生成。  transformer吐出的图像token --> VQ 解码器 --> 像素。

transformer主体是共享的，每个主体的上游和下游则是任务特定的。

输出通过提示词格式进行区分，带`<understand>`标记的路由到SigLIP，带`<generate>`标记的路由到VQ，或者根据任务类型隐式路由。

### 为什么有效

SigLIP特征获取理解损失，因为CLIP风格的预训练针对语义相似度进行调优。模型的感知基线比 Show-o 和 Transfusion 更强，因为输入特征更适合这个任务。

VQ tokens 获取生成损失，因为tokenizer 对重建任务做了调优。图像质量相对于Show-o得到提升，因为VQ 编码能够干净地还原成像素。

共享的transformer主体可以同时看见两种输入分布（SigLIP 和 VQ），然后学会怎么共同工作。断言是：足够大的数据+足够大的参数，模型能够消化这种切换。

### 共享主体的任务

transformer主体处理两种输入分布的相同序列，它的任务在于：
- 对于理解任务，消费SigLIP特征以及文本token，自回归地吐出文本
- 对于生产任务，消费文本token以及可选的图像VQ token，自回归地吐出VQ token。

主体的每个块没有针对特定模态的权重，他就是一个文本风格的transformer，外加两个输入适配器。有意思的是，这也意味着Janus-Pro 的主体可以从一个预训练好的LLM中来。

### InternVL-U

InternVL-U是2026年的后继，它结合了：
- 原生的多模态训练（InternVL3 骨架）
- 解耦的编码器路由（SigLIP输入，VQ+扩散头输出）
- 统一的理解+生成+编辑

InternVL-U 将 Janus-Pro‘s 的架构选择塞进了一个更大的框架。解耦编码器是现在大规模统一模型的默认选择。

### 限制

解耦的编码器增加了架构复杂性。有两个分词器需要训练，两种输入路径需要维护，两种失败模式。

# 开始编码

教学积木：Janus-Pro 核心——**理解用 SigLIP 式连续特征**、**生成用 VQ 离散 token**、**共享 Transformer 主体**、**按任务路由输入适配器**。


## 1. 配置与双编码器：SigLIP 适配器 + 玩具 VQ


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum

import torch
import torch.nn as nn
import torch.nn.functional as F


class JanusTask(str, Enum):
    """任务路由：决定走哪条视觉输入路径。"""

    UNDERSTAND = "understand"
    GENERATE = "generate"


@dataclass
class TinyJanusConfig:
    """Janus-Pro 教学配置（远小于真模型）。"""

    image_size: int = 32
    """输入方图边长。"""

    siglip_dim: int = 32
    """玩具 SigLIP 输出通道维。"""

    siglip_tokens: int = 8
    """理解路径视觉 token 数（真 SigLIP 由网格决定）。"""

    codebook_size: int = 64
    """VQ 码本大小 K。"""

    codebook_dim: int = 32
    """码本向量维。"""

    latent_side: int = 4
    """VQ 特征图边长 → ``N = latent_side**2`` 个图像 token。"""

    text_vocab_size: int = 128
    """文本词表大小。"""

    dim: int = 64
    """共享 Transformer 隐维。"""

    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0

    id_bos: int = 1
    id_eos: int = 2
    id_understand: int = 3
    id_generate: int = 4
    id_img_start: int = 5
    id_img_end: int = 6
    """提示词路由 / 边界符。"""

    @property
    def num_vq_tokens(self) -> int:
        """一张图的 VQ token 数。"""
        return self.latent_side * self.latent_side

    @property
    def img_token_offset(self) -> int:
        """VQ 索引映射到共享离散词表的起点。"""
        return self.text_vocab_size

    @property
    def shared_vocab_size(self) -> int:
        """文本 + VQ 码本（生成头用）。"""
        return self.text_vocab_size + self.codebook_size


class ToySigLIPEncoder(nn.Module):
    """理解路径：图像 → 连续语义 token（SigLIP 占位，无真实对比预训练）。"""

    def __init__(self, cfg: TinyJanusConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.stem = nn.Sequential(
            nn.Conv2d(3, cfg.siglip_dim, kernel_size=4, stride=4),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, cfg.siglip_tokens)),  # (B, C, 1, T)
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            feats: ``(B, T, siglip_dim)`` 连续语义特征。
        """
        cfg = self.cfg
        if images.shape[-2:] != (cfg.image_size, cfg.image_size):
            raise ValueError("unexpected spatial size")
        x = self.stem(images)  # (B, C, 1, T)
        return x.squeeze(2).transpose(1, 2).contiguous()


class SigLIPProjector(nn.Module):
    """2 层 MLP：SigLIP 维 → 主体隐维。"""

    def __init__(self, cfg: TinyJanusConfig) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.siglip_dim, cfg.dim),
            nn.GELU(),
            nn.Linear(cfg.dim, cfg.dim),
        )

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        """
        Args:
            feats: ``(B, T, siglip_dim)``。

        Returns:
            tokens: ``(B, T, D)``。
        """
        return self.net(feats)


class TinyVQTokenizer(nn.Module):
    """生成路径：图像 ↔ 离散码本索引（玩具 VQ-VAE 编码器/解码器）。"""

    def __init__(self, cfg: TinyJanusConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.GELU(),
            nn.Conv2d(32, cfg.codebook_dim, 4, 2, 1),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((cfg.latent_side, cfg.latent_side)),
        )
        self.codebook = nn.Embedding(cfg.codebook_size, cfg.codebook_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(cfg.codebook_dim, 32, 4, 2, 1),
            nn.GELU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            indices: ``(B, N)`` 码本下标。
            quantized: ``(B, N, codebook_dim)`` 量化向量。
        """
        z = self.encoder(images)  # (B, C, h, w)
        B, C, h, w = z.shape
        flat = z.permute(0, 2, 3, 1).reshape(B, h * w, C)
        # 最近邻码本
        dist = (
            flat.pow(2).sum(-1, keepdim=True)
            - 2.0 * flat @ self.codebook.weight.t()
            + self.codebook.weight.pow(2).sum(-1)
        )
        indices = dist.argmin(dim=-1)
        quantized = self.codebook(indices)
        return indices, quantized

    def decode_from_indices(self, indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            indices: ``(B, N)``。

        Returns:
            images: ``(B, 3, H, W)`` 近似重建。
        """
        cfg = self.cfg
        B = indices.size(0)
        q = self.codebook(indices).view(B, cfg.latent_side, cfg.latent_side, cfg.codebook_dim)
        q = q.permute(0, 3, 1, 2).contiguous()
        x = self.decoder(q)
        # 解码器输出空间可能略偏；插值回配置尺寸
        return F.interpolate(x, size=(cfg.image_size, cfg.image_size), mode="bilinear", align_corners=False)

    def forward(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            recon: ``(B, 3, H, W)``。
            indices: ``(B, N)``。
            vq_loss: 码本 + 承诺损失标量。
        """
        indices, quantized = self.encode(images)
        z = self.encoder(images).permute(0, 2, 3, 1).reshape(images.size(0), -1, self.cfg.codebook_dim)
        # stop-grad 风格承诺/码本
        codebook_loss = F.mse_loss(quantized, z.detach())
        commit_loss = F.mse_loss(z, quantized.detach())
        recon = self.decode_from_indices(indices)
        return recon, indices, codebook_loss + 0.25 * commit_loss


print(
    f"dual encoders ready | siglip_T={TinyJanusConfig().siglip_tokens} "
    f"vq_N={TinyJanusConfig().num_vq_tokens}"
)


## 2. 共享主体 + 任务路由嵌入


In [ ]:
class CausalSelfAttention(nn.Module):
    """标准因果多头自注意力。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        if dim % n_heads:
            raise ValueError("dim must divide n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。

        Returns:
            y: ``(B, L, D)``。
        """
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        causal = torch.triu(
            torch.full((L, L), float("-inf"), device=x.device, dtype=x.dtype), 1
        )
        attn = self.drop((attn + causal).softmax(dim=-1))
        h = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(h)


class SharedBlock(nn.Module):
    """共享主体块：无模态专属权重（笔记要点）。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。

        Returns:
            y: ``(B, L, D)``。
        """
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyJanus(nn.Module):
    """
    解耦编码器 + 共享 AR 主体。

    - UNDERSTAND：``SigLIP → MLP`` 连续 token 拼进序列，输出文本 logits
    - GENERATE：VQ 离散 id 走共享嵌入，输出共享词表 logits（可含图像码本）
    """

    def __init__(self, cfg: TinyJanusConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.siglip = ToySigLIPEncoder(cfg)
        self.siglip_proj = SigLIPProjector(cfg)
        self.vq = TinyVQTokenizer(cfg)
        self.text_emb = nn.Embedding(cfg.shared_vocab_size, cfg.dim)
        self.pos = nn.Embedding(512, cfg.dim)
        self.blocks = nn.ModuleList(
            [SharedBlock(cfg.dim, cfg.n_heads, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.norm = nn.LayerNorm(cfg.dim)
        self.lm_head = nn.Linear(cfg.dim, cfg.shared_vocab_size, bias=False)

    def _add_pos(self, x: torch.Tensor) -> torch.Tensor:
        B, L, _ = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        return x + self.pos(pos)

    def encode_vision_for_task(
        self,
        images: torch.Tensor,
        task: JanusTask,
    ) -> tuple[torch.Tensor | None, torch.Tensor | None]:
        """
        按任务路由视觉编码器。

        Args:
            images: ``(B, 3, H, W)``。
            task: ``UNDERSTAND`` 或 ``GENERATE``。

        Returns:
            continuous: ``(B, T, D)`` 或 ``None``（理解路径）。
            vq_indices: ``(B, N)`` 或 ``None``（生成路径条件图 / 重建）。
        """
        if task is JanusTask.UNDERSTAND:
            feats = self.siglip(images)
            return self.siglip_proj(feats), None
        indices, _ = self.vq.encode(images)
        return None, indices

    def forward_understand(
        self,
        images: torch.Tensor,
        text_ids: torch.Tensor,
    ) -> torch.Tensor:
        """
        VQA / 理解：图像走 SigLIP，再拼文本，因果预测文本。

        布局：``[<understand>] + vision_tokens + text_ids``。

        Args:
            images: ``(B, 3, H, W)``。
            text_ids: ``(B, Lt)`` 含问题与答案的文本（teacher forcing）。

        Returns:
            logits: ``(B, 1+T+Lt, V)`` 全序列 logits（通常对文本段算 CE）。
        """
        cfg = self.cfg
        B = images.size(0)
        vision, _ = self.encode_vision_for_task(images, JanusTask.UNDERSTAND)
        assert vision is not None
        route = self.text_emb(
            torch.full((B, 1), cfg.id_understand, device=images.device, dtype=torch.long)
        )
        text_h = self.text_emb(text_ids)
        x = self._add_pos(torch.cat([route, vision, text_h], dim=1))
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.norm(x))

    def forward_generate(
        self,
        text_ids: torch.Tensor,
        image_indices: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        T2I / 生成：文本（+ 可选已有 VQ token）走离散嵌入，预测下一共享词表 token。

        布局：``[<generate>] + text + [<image>] + vq_ids + [</image>]``（若提供图像）。

        Args:
            text_ids: ``(B, Lt)``。
            image_indices: ``(B, N)`` 码本下标；``None`` 表示纯前缀（生成起始）。

        Returns:
            logits: ``(B, L, V)``。
        """
        cfg = self.cfg
        B = text_ids.size(0)
        device = text_ids.device
        route = torch.full((B, 1), cfg.id_generate, device=device, dtype=torch.long)
        parts: list[torch.Tensor] = [route, text_ids]
        if image_indices is not None:
            start = torch.full((B, 1), cfg.id_img_start, device=device, dtype=torch.long)
            end = torch.full((B, 1), cfg.id_img_end, device=device, dtype=torch.long)
            img_ids = image_indices + cfg.img_token_offset
            parts.extend([start, img_ids, end])
        ids = torch.cat(parts, dim=1)
        x = self._add_pos(self.text_emb(ids))
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(self.norm(x))

    def understand_loss(self, images: torch.Tensor, text_ids: torch.Tensor) -> torch.Tensor:
        """
        理解任务 CE：只在文本段做 next-token（视觉位不作为标签）。

        Args:
            images: ``(B, 3, H, W)``。
            text_ids: ``(B, Lt)``。

        Returns:
            loss: 标量。
        """
        cfg = self.cfg
        logits = self.forward_understand(images, text_ids)
        # 序列：route(1) + vision(T) + text(Lt)；对 text 做移位 CE
        t0 = 1 + cfg.siglip_tokens
        text_logits = logits[:, t0 - 1 : -1, :]  # 预测 text 各位
        return F.cross_entropy(
            text_logits.reshape(-1, text_logits.size(-1)),
            text_ids.reshape(-1),
        )

    def generate_loss(
        self,
        text_ids: torch.Tensor,
        image_indices: torch.Tensor,
    ) -> torch.Tensor:
        """
        生成任务 CE：在整段离散序列上做标准 NTP（含 VQ token）。

        Args:
            text_ids: ``(B, Lt)``。
            image_indices: ``(B, N)``。

        Returns:
            loss: 标量。
        """
        cfg = self.cfg
        B = text_ids.size(0)
        device = text_ids.device
        route = torch.full((B, 1), cfg.id_generate, device=device, dtype=torch.long)
        start = torch.full((B, 1), cfg.id_img_start, device=device, dtype=torch.long)
        end = torch.full((B, 1), cfg.id_img_end, device=device, dtype=torch.long)
        img_ids = image_indices + cfg.img_token_offset
        full = torch.cat([route, text_ids, start, img_ids, end], dim=1)
        logits = self.forward_generate(text_ids, image_indices)
        return F.cross_entropy(
            logits[:, :-1].reshape(-1, logits.size(-1)),
            full[:, 1:].reshape(-1),
        )


print("TinyJanus shared backbone ready")


## 3. 推理示意：理解出文本 / 生成出 VQ 再解码


In [ ]:
@torch.no_grad()
def greedy_answer(
    model: TinyJanus,
    images: torch.Tensor,
    question_ids: torch.Tensor,
    max_new_tokens: int = 8,
) -> torch.Tensor:
    """
    理解路径贪心续写文本。

    Args:
        model: Janus 玩具模型。
        images: ``(1, 3, H, W)``。
        question_ids: ``(1, Lq)``。
        max_new_tokens: 最多新生成 token 数。

    Returns:
        answer_ids: ``(1, L)`` 含问题与续写。
    """
    cfg = model.cfg
    ids = question_ids.clone()
    for _ in range(max_new_tokens):
        logits = model.forward_understand(images, ids)
        # 最后一个位置对应预测下一个文本 token
        next_id = int(logits[:, -1, : cfg.text_vocab_size].argmax(dim=-1).item())
        ids = torch.cat([ids, torch.tensor([[next_id]], device=ids.device)], dim=1)
        if next_id == cfg.id_eos:
            break
    return ids


@torch.no_grad()
def greedy_generate_image(
    model: TinyJanus,
    prompt_ids: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    生成路径：在 ``<image>`` 后自回归吐满 ``N`` 个 VQ token，再解码像素。

    Args:
        model: 模型。
        prompt_ids: ``(1, Lt)`` 文本提示。

    Returns:
        indices: ``(1, N)``。
        images: ``(1, 3, H, W)``。
    """
    cfg = model.cfg
    device = prompt_ids.device
    # 先写到 <image>，再逐个采图像 token
    B = 1
    route = torch.full((B, 1), cfg.id_generate, device=device, dtype=torch.long)
    start = torch.full((B, 1), cfg.id_img_start, device=device, dtype=torch.long)
    ids = torch.cat([route, prompt_ids, start], dim=1)
    img_buf: list[int] = []
    for _ in range(cfg.num_vq_tokens):
        x = model._add_pos(model.text_emb(ids))
        for blk in model.blocks:
            x = blk(x)
        logits = model.lm_head(model.norm(x))[:, -1, :]
        # 限制在图像码本区间
        span = logits[:, cfg.img_token_offset : cfg.img_token_offset + cfg.codebook_size]
        local = int(span.argmax(dim=-1).item())
        tok = cfg.img_token_offset + local
        img_buf.append(local)
        ids = torch.cat([ids, torch.tensor([[tok]], device=device)], dim=1)
    indices = torch.tensor([img_buf], device=device, dtype=torch.long)
    images = model.vq.decode_from_indices(indices)
    return indices, images


def route_from_prompt_token(first_id: int, cfg: TinyJanusConfig) -> JanusTask:
    """
    根据首个特殊标记做显式路由。

    Args:
        first_id: 提示词首 token。
        cfg: 配置。

    Returns:
        task: ``UNDERSTAND`` 或 ``GENERATE``。
    """
    if first_id == cfg.id_understand:
        return JanusTask.UNDERSTAND
    if first_id == cfg.id_generate:
        return JanusTask.GENERATE
    raise ValueError(f"unknown route token: {first_id}")


print("inference helpers ready")


## 4. 冒烟测试


In [ ]:
def smoke_test() -> None:
    """验证双编码器路由、共享主体双损失、生成解码。"""
    torch.manual_seed(0)
    cfg = TinyJanusConfig()
    model = TinyJanus(cfg)

    images = torch.randn(2, 3, cfg.image_size, cfg.image_size)
    text = torch.randint(7, cfg.text_vocab_size, (2, 5))

    print("=== encoder routing ===")
    cont, vq_none = model.encode_vision_for_task(images, JanusTask.UNDERSTAND)
    none_c, vq_idx = model.encode_vision_for_task(images, JanusTask.GENERATE)
    assert cont is not None and vq_none is None
    assert none_c is None and vq_idx is not None
    print(f"siglip_tokens={tuple(cont.shape)} vq_indices={tuple(vq_idx.shape)}")
    assert cont.shape == (2, cfg.siglip_tokens, cfg.dim)
    assert vq_idx.shape == (2, cfg.num_vq_tokens)

    print("\n=== understand loss ===")
    u_loss = model.understand_loss(images, text)
    print(f"understand_loss={u_loss.item():.4f}")
    u_loss.backward()
    model.zero_grad(set_to_none=True)

    print("\n=== generate loss ===")
    g_loss = model.generate_loss(text, vq_idx.detach())
    print(f"generate_loss={g_loss.item():.4f}")
    g_loss.backward()
    assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())

    print("\n=== VQ recon ===")
    recon, indices, vq_loss = model.vq(images)
    print(f"recon={tuple(recon.shape)} vq_loss={vq_loss.item():.4f}")
    assert recon.shape == images.shape

    print("\n=== greedy generate image ===")
    model.eval()
    prompt = torch.randint(7, cfg.text_vocab_size, (1, 3))
    idx, rendered = greedy_generate_image(model, prompt)
    print(f"indices={tuple(idx.shape)} rendered={tuple(rendered.shape)}")
    assert rendered.shape == (1, 3, cfg.image_size, cfg.image_size)

    print("\n=== route token ===")
    assert route_from_prompt_token(cfg.id_understand, cfg) is JanusTask.UNDERSTAND
    assert route_from_prompt_token(cfg.id_generate, cfg) is JanusTask.GENERATE

    print("SMOKE TEST OK")


smoke_test()
